# Task 2: Exploratory Data Analysis

This notebook performs comprehensive exploratory data analysis on the enriched Ethiopia Financial Inclusion dataset.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', 1000)

# Set style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

# Load enriched data
data_path = '../data/processed/ethiopia_fi_unified_data_enriched.csv'
impact_path = '../data/processed/impact_links_enriched.csv'

data_df = pd.read_csv(data_path)
impact_df = pd.read_csv(impact_path)

# Convert dates
data_df['observation_date'] = pd.to_datetime(data_df['observation_date'], errors='coerce')
impact_df['observation_date'] = pd.to_datetime(impact_df['observation_date'], errors='coerce')

print("Enriched data loaded successfully!")
print(f"Data records: {len(data_df)}")
print(f"Impact links: {len(impact_df)}")

## 1. Dataset Overview

In [ ]:
# Summarize dataset by record_type, pillar, and source_type
print("=== DATASET SUMMARY ===")
print(f"\nTotal Records: {len(data_df)}")
print(f"\nBy Record Type:")
print(data_df['record_type'].value_counts())

print(f"\nBy Pillar:")
print(data_df['pillar'].value_counts())

print(f"\nBy Source Type:")
print(data_df['source_type'].value_counts())

print(f"\nBy Confidence:")
print(data_df['confidence'].value_counts())

In [ ]:
# Create visualization of record distribution
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Record type distribution
data_df['record_type'].value_counts().plot(kind='bar', ax=axes[0,0], color='steelblue')
axes[0,0].set_title('Distribution by Record Type')
axes[0,0].set_ylabel('Count')
axes[0,0].tick_params(axis='x', rotation=45)

# Pillar distribution
data_df['pillar'].value_counts().plot(kind='bar', ax=axes[0,1], color='coral')
axes[0,1].set_title('Distribution by Pillar')
axes[0,1].set_ylabel('Count')
axes[0,1].tick_params(axis='x', rotation=45)

# Source type distribution
data_df['source_type'].value_counts().plot(kind='bar', ax=axes[1,0], color='lightgreen')
axes[1,0].set_title('Distribution by Source Type')
axes[1,0].set_ylabel('Count')
axes[1,0].tick_params(axis='x', rotation=45)

# Confidence distribution
data_df['confidence'].value_counts().plot(kind='bar', ax=axes[1,1], color='plum')
axes[1,1].set_title('Distribution by Confidence Level')
axes[1,1].set_ylabel('Count')
axes[1,1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('../reports/figures/dataset_overview.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Temporal coverage visualization
observations = data_df[data_df['record_type'] == 'observation']

# Get unique indicators and their date ranges
indicator_coverage = []
for indicator in observations['indicator_code'].dropna().unique():
    subset = observations[observations['indicator_code'] == indicator]
    if not subset['observation_date'].isna().all():
        indicator_coverage.append({
            'indicator': indicator,
            'pillar': subset['pillar'].mode()[0] if not subset['pillar'].isna().all() else 'Unknown',
            'start': subset['observation_date'].min(),
            'end': subset['observation_date'].max(),
            'count': len(subset)
        })

coverage_df = pd.DataFrame(indicator_coverage)
coverage_df = coverage_df.sort_values('pillar')

# Create temporal coverage plot
fig, ax = plt.subplots(figsize=(12, 8))

for idx, row in coverage_df.iterrows():
    ax.barh(row['indicator'], 
            (row['end'] - row['start']).days / 365.25,
            left=row['start'],
            height=0.6,
            alpha=0.7,
            label=row['pillar'])

ax.set_xlabel('Year')
ax.set_title('Temporal Coverage of Indicators')
ax.grid(axis='x', alpha=0.3)

# Format x-axis as years
ax.xaxis_date()
ax.xaxis.set_major_formatter(plt.matplotlib.dates.DateFormatter('%Y'))

plt.tight_layout()
plt.savefig('../reports/figures/temporal_coverage.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n=== INDICATOR COVERAGE SUMMARY ===")
print(coverage_df.to_string())

In [ ]:
# Data quality assessment
print("=== DATA QUALITY ASSESSMENT ===")

# Missing values analysis
print("\nMissing values by column:")
missing = data_df.isnull().sum()
missing_pct = (missing / len(data_df)) * 100
missing_summary = pd.DataFrame({'Count': missing, 'Percentage': missing_pct})
print(missing_summary[missing_summary['Count'] > 0])

# Confidence analysis
print("\n=== CONFIDENCE ANALYSIS ===")
confidence_by_type = pd.crosstab(data_df['record_type'], data_df['confidence'])
print(confidence_by_type)

# Identify sparse indicators
print("\n=== SPARSE INDICATORS (≤2 records) ===")
sparse_indicators = coverage_df[coverage_df['count'] <= 2]
print(sparse_indicators[['indicator', 'count', 'pillar']])

## 2. Access Analysis - Account Ownership Trajectory

In [ ]:
# Filter account ownership data
access_data = observations[
    (observations['indicator_code'] == 'ACC_OWNERSHIP') & 
    (observations['gender'] == 'all')
].sort_values('observation_date')

print("=== ACCOUNT OWNERSHIP DATA ===")
print(access_data[['observation_date', 'value_numeric', 'source_name']])

In [ ]:
# Plot account ownership trajectory
fig, ax = plt.subplots(figsize=(12, 6))

ax.plot(access_data['observation_date'], access_data['value_numeric'], 
        marker='o', linewidth=2, markersize=8, color='steelblue')

# Add data labels
for idx, row in access_data.iterrows():
    ax.annotate(f"{row['value_numeric']:.1f}%",
                (row['observation_date'], row['value_numeric']),
                textcoords="offset points", xytext=(0,10), ha='center')

ax.set_xlabel('Year')
ax.set_ylabel('Account Ownership Rate (%)')
ax.set_title('Ethiopia Account Ownership Trajectory (2011-2024)')
ax.set_ylim(0, 60)
ax.grid(True, alpha=0.3)

# Add target line
ax.axhline(y=60, color='red', linestyle='--', alpha=0.5, label='NFIS-II Target (60%)')
ax.legend()

plt.tight_layout()
plt.savefig('../reports/figures/account_ownership_trajectory.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Calculate growth rates
access_data = access_data.copy()
access_data['year'] = access_data['observation_date'].dt.year
access_data['growth_pp'] = access_data['value_numeric'].diff()
access_data['growth_pct'] = (access_data['value_numeric'].pct_change() * 100).round(1)

print("=== ACCOUNT OWNERSHIP GROWTH RATES ===")
print(access_data[['year', 'value_numeric', 'growth_pp', 'growth_pct']])

# Average growth rates by period
print("\n=== AVERAGE GROWTH BY PERIOD ===")
print(f"2011-2014: +{(access_data.iloc[1]['value_numeric'] - access_data.iloc[0]['value_numeric']):.1f} pp over 3 years")
print(f"2014-2017: +{(access_data.iloc[2]['value_numeric'] - access_data.iloc[1]['value_numeric']):.1f} pp over 3 years")
print(f"2017-2021: +{(access_data.iloc[3]['value_numeric'] - access_data.iloc[2]['value_numeric']):.1f} pp over 4 years")
print(f"2021-2024: +{(access_data.iloc[4]['value_numeric'] - access_data.iloc[3]['value_numeric']):.1f} pp over 3 years")

In [ ]:
# Gender gap analysis
gender_data = observations[
    (observations['indicator_code'] == 'ACC_OWNERSHIP') & 
    (observations['gender'].isin(['male', 'female']))
].sort_values(['observation_date', 'gender'])

if len(gender_data) > 0:
    print("=== GENDER DISAGGREGATED DATA ===")
    print(gender_data[['observation_date', 'gender', 'value_numeric']])
    
    # Calculate gender gap
    male_data = gender_data[gender_data['gender'] == 'male']
    female_data = gender_data[gender_data['gender'] == 'female']
    
    # Create gender comparison plot
    fig, ax = plt.subplots(figsize=(10, 6))
    
    width = 0.35
    years = male_data['observation_date'].dt.year.unique()
    
    male_values = male_data['value_numeric'].values
    female_values = female_data['value_numeric'].values
    
    x = np.arange(len(years))
    
    bars1 = ax.bar(x - width/2, male_values, width, label='Male', color='steelblue')
    bars2 = ax.bar(x + width/2, female_values, width, label='Female', color='coral')
    
    ax.set_xlabel('Year')
    ax.set_ylabel('Account Ownership Rate (%)')
    ax.set_title('Gender Gap in Account Ownership')
    ax.set_xticks(x)
    ax.set_xticklabels(years)
    ax.legend()
    ax.grid(axis='y', alpha=0.3)
    
    # Add gap labels
    for i in range(len(years)):
        gap = male_values[i] - female_values[i]
        ax.annotate(f"Gap: {gap:.1f}pp",
                    (x[i], max(male_values[i], female_values[i]) + 2),
                    textcoords="offset points", xytext=(0,5), ha='center', fontsize=9)
    
    plt.tight_layout()
    plt.savefig('../reports/figures/gender_gap_account_ownership.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    print(f"\nGender Gap in 2021: {male_values[0] - female_values[0]:.1f} percentage points")
else:
    print("No gender-disaggregated data available for account ownership")

In [ ]:
# Investigate 2021-2024 slowdown
print("=== 2021-2024 SLOWDOWN ANALYSIS ===")
print("\nContext:")
print("- Account ownership grew only +3pp (46% to 49%) from 2021 to 2024")
print("- This is the slowest growth period despite massive mobile money expansion")
print("- Telebirr grew to 54M+ users since 2021 launch")
print("- M-Pesa entered in 2023 with 10M+ users")

print("\nPotential Explanations:")
print("1. Registered vs. Active Gap: Mobile money accounts may not translate to 'account ownership' as defined by Findex")
print("2. Definition Mismatch: Findex defines account ownership broadly (bank OR mobile money)")
print("3. User Overlap: Same users may have multiple mobile money accounts")
print("4. Dormant Accounts: High registration but low active usage")
print("5. Survey Timing: 2024 survey may have captured early adoption phase")
print("6. Traditional Bank Growth: Traditional banking may have stagnated while mobile money grew")

## 3. Usage Analysis - Digital Payments

In [ ]:
# Mobile money account penetration
mm_data = observations[
    observations['indicator_code'] == 'ACC_MM_ACCOUNT'
].sort_values('observation_date')

print("=== MOBILE MONEY ACCOUNT DATA ===")
print(mm_data[['observation_date', 'value_numeric', 'source_name']])

if len(mm_data) > 0:
    fig, ax = plt.subplots(figsize=(10, 6))
    
    ax.plot(mm_data['observation_date'], mm_data['value_numeric'], 
            marker='o', linewidth=2, markersize=8, color='green')
    
    for idx, row in mm_data.iterrows():
        ax.annotate(f"{row['value_numeric']:.1f}%",
                    (row['observation_date'], row['value_numeric']),
                    textcoords="offset points", xytext=(0,10), ha='center')
    
    ax.set_xlabel('Year')
    ax.set_ylabel('Mobile Money Account Ownership (%)')
    ax.set_title('Mobile Money Account Penetration (2021-2024)')
    ax.set_ylim(0, 15)
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('../reports/figures/mobile_money_penetration.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    print(f"\nGrowth: {(mm_data.iloc[-1]['value_numeric'] - mm_data.iloc[0]['value_numeric']):.1f} pp from 2021 to 2024")
    print(f"Growth Rate: {((mm_data.iloc[-1]['value_numeric'] / mm_data.iloc[0]['value_numeric'] - 1) * 100):.1f}%")

In [ ]:
# Digital payment adoption
digital_payment_data = observations[
    observations['indicator_code'] == 'USG_DIGITAL_PAYMENT'
].sort_values('observation_date')

print("=== DIGITAL PAYMENT ADOPTION DATA ===")
print(digital_payment_data[['observation_date', 'value_numeric', 'source_name']])

if len(digital_payment_data) > 0:
    print(f"\nDigital Payment Adoption Rate: {digital_payment_data.iloc[0]['value_numeric']:.1f}%")
    print("This is the core usage indicator for forecasting")

In [ ]:
# P2P transaction analysis
p2p_data = observations[
    observations['indicator_code'].str.contains('P2P', na=False)
].sort_values('observation_date')

print("=== P2P TRANSACTION DATA ===")
print(p2p_data[['indicator_code', 'observation_date', 'value_numeric', 'unit']])

## 4. Infrastructure and Enablers Analysis

In [ ]:
# Infrastructure indicators
infrastructure_indicators = ['ACC_4G_COV', 'ACC_SMARTPHONE', 'ACC_ATM_DENSITY', 'ACC_BRANCH_DENSITY', 'ACC_AGENT_DENSITY']
infra_data = observations[
    observations['indicator_code'].isin(infrastructure_indicators)
].sort_values(['indicator_code', 'observation_date'])

print("=== INFRASTRUCTURE DATA ===")
print(infra_data[['indicator_code', 'observation_date', 'value_numeric', 'unit']])

In [ ]:
# 4G coverage trend
four_g_data = observations[observations['indicator_code'] == 'ACC_4G_COV'].sort_values('observation_date')

if len(four_g_data) > 0:
    fig, ax = plt.subplots(figsize=(10, 6))
    
    ax.plot(four_g_data['observation_date'], four_g_data['value_numeric'], 
            marker='o', linewidth=2, markersize=8, color='purple')
    
    for idx, row in four_g_data.iterrows():
        ax.annotate(f"{row['value_numeric']:.1f}%",
                    (row['observation_date'], row['value_numeric']),
                    textcoords="offset points", xytext=(0,10), ha='center')
    
    ax.set_xlabel('Year')
    ax.set_ylabel('4G Population Coverage (%)')
    ax.set_title('4G Network Coverage Expansion')
    ax.set_ylim(0, 80)
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('../reports/figures/4g_coverage.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    print(f"\n4G Coverage Growth: {(four_g_data.iloc[-1]['value_numeric'] - four_g_data.iloc[0]['value_numeric']):.1f} pp")
    print(f"Growth Rate: {((four_g_data.iloc[-1]['value_numeric'] / four_g_data.iloc[0]['value_numeric'] - 1) * 100):.1f}%")

In [ ]:
# Infrastructure comparison
infra_summary = infra_data.groupby('indicator_code').agg({
    'value_numeric': 'last',
    'unit': 'first'
}).reset_index()

print("=== INFRASTRUCTURE SUMMARY (LATEST VALUES) ===")
print(infra_summary)

## 5. Event Timeline and Visual Analysis

In [ ]:
# Extract events
events = data_df[data_df['record_type'] == 'event'].sort_values('observation_date')

print("=== EVENT TIMELINE ===")
for idx, row in events.iterrows():
    print(f"{row['observation_date'].date()}: {row['indicator']} ({row['category']})")

In [ ]:
# Create event timeline visualization
fig, ax = plt.subplots(figsize=(14, 8))

# Define colors for event categories
category_colors = {
    'product_launch': 'steelblue',
    'market_entry': 'coral',
    'policy': 'lightgreen',
    'regulation': 'purple',
    'infrastructure': 'orange',
    'partnership': 'pink',
    'milestone': 'red',
    'pricing': 'brown'
}

# Plot events
y_positions = range(len(events))
for idx, (event_idx, row) in enumerate(events.iterrows()):
    color = category_colors.get(row['category'], 'gray')
    ax.scatter(row['observation_date'], idx, 
               s=200, c=color, alpha=0.7, edgecolors='black')
    ax.annotate(row['indicator'],
                (row['observation_date'], idx),
                xytext=(10, 0), textcoords='offset points',
                fontsize=9, va='center')
    ax.annotate(row['category'],
                (row['observation_date'], idx),
                xytext=(10, -15), textcoords='offset points',
                fontsize=8, va='center', style='italic', color='gray')

ax.set_yticks(y_positions)
ax.set_yticklabels([f"{row['record_id']}" for idx, row in events.iterrows()])
ax.set_xlabel('Date')
ax.set_title('Ethiopia Financial Inclusion Event Timeline')
ax.grid(axis='x', alpha=0.3)

# Add legend
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=color, label=cat) 
                   for cat, color in category_colors.items()]
ax.legend(handles=legend_elements, loc='upper left', bbox_to_anchor=(1, 1))

plt.tight_layout()
plt.savefig('../reports/figures/event_timeline.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Overlay events on account ownership chart
fig, ax = plt.subplots(figsize=(14, 6))

# Plot account ownership
ax.plot(access_data['observation_date'], access_data['value_numeric'], 
        marker='o', linewidth=2, markersize=8, color='steelblue', label='Account Ownership')

# Add vertical lines for key events
key_events = [
    ('2020-03-15', 'NBE MM Regulation', 'regulation'),
    ('2021-05-17', 'Telebirr Launch', 'product_launch'),
    ('2021-09-01', 'NFIS-II Launch', 'policy'),
    ('2022-08-01', 'Safaricom Entry', 'market_entry'),
    ('2023-01-15', '4G Expansion', 'infrastructure'),
    ('2023-08-01', 'M-Pesa Launch', 'product_launch'),
    ('2024-01-01', 'Fayda Rollout', 'infrastructure'),
    ('2024-07-29', 'FX Reform', 'policy'),
    ('2024-10-01', 'P2P > ATM Milestone', 'milestone')
]

for date_str, label, category in key_events:
    event_date = pd.to_datetime(date_str)
    color = category_colors.get(category, 'gray')
    ax.axvline(x=event_date, color=color, linestyle='--', alpha=0.5)
    ax.annotate(label, (event_date, ax.get_ylim()[1]),
                rotation=90, ha='right', va='top', fontsize=8,
                bbox=dict(boxstyle='round,pad=0.3', facecolor=color, alpha=0.3))

ax.set_xlabel('Year')
ax.set_ylabel('Account Ownership Rate (%)')
ax.set_title('Account Ownership with Key Events Overlay')
ax.set_ylim(0, 60)
ax.legend(loc='upper left')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../reports/figures/account_ownership_with_events.png', dpi=300, bbox_inches='tight')
plt.show()

## 6. Correlation Analysis

In [ ]:
# Prepare data for correlation analysis
# Focus on ACCESS and USAGE indicators with numeric values
analysis_indicators = [
    'ACC_OWNERSHIP', 'ACC_MM_ACCOUNT', 'ACC_4G_COV', 'ACC_SMARTPHONE',
    'USG_DIGITAL_PAYMENT', 'USG_P2P_COUNT'
]

corr_data = observations[
    observations['indicator_code'].isin(analysis_indicators)
].copy()

# Create pivot table for correlation
corr_pivot = corr_data.pivot_table(
    index='observation_date',
    columns='indicator_code',
    values='value_numeric'
)

print("=== CORRELATION MATRIX DATA ===")
print(corr_pivot)

# Calculate correlation matrix
corr_matrix = corr_pivot.corr()

print("\n=== CORRELATION MATRIX ===")
print(corr_matrix.round(2))

In [ ]:
# Create correlation heatmap
fig, ax = plt.subplots(figsize=(10, 8))

sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', center=0,
            square=True, linewidths=1, cbar_kws={"shrink": 0.8}, ax=ax)

ax.set_title('Correlation Matrix of Key Indicators')
plt.tight_layout()
plt.savefig('../reports/figures/correlation_matrix.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Analyze impact link relationships
print("=== IMPACT LINK ANALYSIS ===")
print(f"Total impact links: {len(impact_df)}")

# Impact links by pillar
print("\n=== IMPACTS BY PILLAR ===")
print(impact_df['pillar'].value_counts())

# Impact links by direction
print("\n=== IMPACTS BY DIRECTION ===")
print(impact_df['impact_direction'].value_counts())

# Impact links by magnitude
print("\n=== IMPACTS BY MAGNITUDE ===")
print(impact_df['impact_magnitude'].value_counts())

In [ ]:
# Event-indicator relationship summary
event_impact_summary = impact_df.groupby(['parent_id', 'pillar']).agg({
    'related_indicator': 'count',
    'impact_direction': lambda x: x.mode()[0] if len(x.mode()) > 0 else x.iloc[0]
}).reset_index()

print("=== EVENT-INDICATOR RELATIONSHIP SUMMARY ===")
print(event_impact_summary)

## 7. Key Insights Summary

In [ ]:
print("""=== KEY INSIGHTS FROM EDA ===

1. ACCOUNT OWNERSHIP TRAJECTORY
   - Ethiopia's account ownership grew from 14% (2011) to 49% (2024)
   - Strong growth periods: 2011-2014 (+8pp), 2014-2017 (+13pp), 2017-2021 (+11pp)
   - SLOWDOWN: 2021-2024 (+3pp) despite massive mobile money expansion
   - Gender gap in 2021: 20pp (56% male vs 36% female)

2. MOBILE MONEY EXPANSION
   - Mobile money accounts doubled from 4.7% (2021) to 9.45% (2024)
   - Telebirr: 54M+ users since 2021 launch
   - M-Pesa: 10M+ users since 2023 entry
   - Despite growth, mobile money accounts still only 9.45% of adult population

3. INFRASTRUCTURE ENABLERS
   - 4G coverage nearly doubled from 37.5% (2023) to 70.8% (2025)
   - Smartphone penetration at 35% (2023)
   - Agent density: 45.2 per 100k adults (2024)
   - ATM density: 12.5 per 100k adults (2023)

4. DIGITAL PAYMENT ADOPTION
   - 35% of adults made or received digital payments (2024)
   - This is significantly higher than mobile money account ownership (9.45%)
   - Suggests digital payments through other channels (banking apps, cards)

5. EVENT TIMING AND IMPACTS
   - NBE MM Regulation (Mar 2020) preceded Telebirr launch (May 2021)
   - 4G expansion (Jan 2023) aligns with coverage growth
   - Multiple product launches and market entries 2021-2023
   - P2P surpassed ATM transactions (Oct 2024) - key milestone

6. DATA GAPS AND LIMITATIONS
   - Sparse data: Many indicators have only 1-2 data points
   - Limited temporal coverage: Most data post-2020
   - No regional disaggregation
   - Limited gender-disaggregated data
   - Missing quarterly data for trend analysis

7. HYPOTHESES FOR FORECASTING
   - Account ownership slowdown may be temporary (survey timing, definition issues)
   - Infrastructure improvements (4G, Fayda) may accelerate future growth
   - Competition (Safaricom, M-Pesa) may drive innovation and adoption
   - Agent network expansion critical for cash-in/cash-out services
   - Digital ID (Fayda) may reduce gender gap and increase account opening
""")

## 8. Data Quality Assessment

In [ ]:
print("""=== DATA QUALITY ASSESSMENT ===

STRENGTHS:
1. High confidence data: 93% of records have 'high' confidence
2. Authoritative sources: Global Findex, NBE, GSMA, operators
3. Unified schema: Consistent structure across record types
4. Event modeling: Explicit event-indicator relationships via impact_links
5. Schema compliance: Events correctly have no pillar assignments

WEAKNESSES:
1. Sparse temporal coverage: Most indicators have 1-3 data points
2. Limited historical data: Account ownership only back to 2011
3. No regional disaggregation: All data at national level
4. Limited gender data: Only account ownership has gender breakdown
5. Missing infrastructure data: No POS, QR merchant, literacy data
6. No transaction volumes: Limited data on actual usage patterns
7. Survey frequency: Findex only every 3 years

FORECASTING IMPLICATIONS:
1. Limited historical data makes time series modeling challenging
2. Event-driven modeling may be more appropriate than pure time series
3. Scenario analysis needed to account for uncertainty
4. External data may be needed for leading indicators
5. Expert judgment important for impact estimation

RECOMMENDATIONS:
1. Use event-augmented models rather than pure time series
2. Incorporate scenario analysis for uncertainty quantification
3. Use comparable country evidence for impact estimation
4. Focus on ACCESS and USAGE as primary forecasting targets
5. Document assumptions and limitations clearly
""")